<a href="https://colab.research.google.com/github/kimjiwoo2/Pill-agent/blob/develop/notebooks/jiwoo/05_jw_cls_full_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **0. Overview**

### **full (single+combination) (v1)**

- optimized split 사용 (이미 전처리/레이블 완료)
- 미리 저장된 crop 이미지 사용 → 실시간 전처리 없음
- train 18,897 / val 3,512
- color/shape 분포 불균형 (val에 일부 클래스 누락)
- **ConvNeXt-tiny** 학습 시도
- epoch당 배치 591개로 학습 시간 과다, 런타임 끊김 발생
-  combination crop 품질 문제로 오히려 학습 성능 저하 가능성 있음
- 결론: single 12,000장 기준 best model 채택, 추후 데이터/split 전략 개선 검토

# **1. Import**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
import pandas as pd
import numpy as np
!pip install pymysql
import pymysql
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
import pickle
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import cv2
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
import torch
import os
from PIL import Image
from tqdm import tqdm

# **2. Manifest 로드 및 이미지 저장**

In [ ]:
# 전처리 및 저장 함수
def preprocess_and_save(df, zip_path, save_dir):
    failed = []
    with zipfile.ZipFile(zip_path, 'r') as z:
        for idx, row in tqdm(df.iterrows(), total=len(df)):
            save_path = os.path.join(save_dir, row['image_file'])
            if os.path.exists(save_path):
                continue
            try:
                with z.open(row['zip_path']) as f:
                    img = Image.open(f).convert('RGB')

                img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

                # AWB
                result = img_cv.copy().astype(np.float32)
                for i in range(3):
                    channel = result[:, :, i]
                    result[:, :, i] = channel * (128.0 / (np.mean(channel) + 1e-6))
                img_cv = np.clip(result, 0, 255).astype(np.uint8)

                # Bilateral Filter
                img_cv = cv2.bilateralFilter(img_cv, d=5, sigmaColor=30, sigmaSpace=30)

                # bbox crop
                x, y, w, h = int(row['bbox_x']), int(row['bbox_y']), int(row['bbox_w']), int(row['bbox_h'])
                pad_x, pad_y = int(w * 0.1), int(h * 0.1)
                ih, iw = img_cv.shape[:2]
                x1, y1 = max(0, x - pad_x), max(0, y - pad_y)
                x2, y2 = min(iw, x + w + pad_x), min(ih, y + h + pad_y)
                cropped = img_cv[y1:y2, x1:x2]

                # CLAHE
                clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
                for i in range(3):
                    cropped[:, :, i] = clahe.apply(cropped[:, :, i])

                # 저장
                cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
                Image.fromarray(cropped_rgb).save(save_path)

            except Exception as e:
                failed.append((row['image_file'], str(e)))

    print(f"완료. 실패: {len(failed)}개")
    return failed

In [ ]:
# 경로 설정
manifest_zip_path = '/content/drive/MyDrive/ToBigs/2425/Pillot/dataset/pilliot_15k_optimized_leakage_free_split.zip'
dataset_path = '/content/drive/MyDrive/ToBigs/2425/Pillot/dataset/pilliot_15k_v1_final.zip'
crop_save_dir = '/content/drive/MyDrive/ToBigs/2425/Pillot/dataset/jiwoo/cropped_images/'
os.makedirs(crop_save_dir, exist_ok=True)

# manifest 로드
with zipfile.ZipFile(manifest_zip_path, 'r') as z:
    with z.open('pilliot_15k_optimized_leakage_free_split/manifests/object_manifest_train_with_attributes.csv') as f:
        df_train = pd.read_csv(f)
    with z.open('pilliot_15k_optimized_leakage_free_split/manifests/object_manifest_val_with_attributes.csv') as f:
        df_val = pd.read_csv(f)

print(f"train: {len(df_train)}행 / val: {len(df_val)}행")
print(f"item_seq overlap: {len(set(df_train['item_seq']) & set(df_val['item_seq']))}")

# zip 내부 파일명 → 경로 매핑
with zipfile.ZipFile(dataset_path, 'r') as z:
    path_map = {}
    for name in z.namelist():
        if name.endswith('.png') or name.endswith('.jpg'):
            filename = name.split('/')[-1]
            path_map[filename] = name

print(f"zip 내 이미지 수: {len(path_map)}")

# 전체 이미지 목록
df_all = pd.concat([df_train, df_val]).drop_duplicates(subset='image_file').reset_index(drop=True)
df_all['zip_path'] = df_all['image_file'].map(path_map)
print(f"전체 이미지: {len(df_all)}개")
print(f"매핑 성공: {df_all['zip_path'].notna().sum()}")
print(f"매핑 실패: {df_all['zip_path'].isna().sum()}")

# 미저장 이미지만 필터링
saved = set(os.listdir(crop_save_dir))
df_todo = df_all[~df_all['image_file'].isin(saved) & df_all['zip_path'].notna()].copy()
print(f"\n저장 완료: {len(saved)}개")
print(f"저장할 이미지: {len(df_todo)}개")

# 전처리 및 저장
failed = preprocess_and_save(df_todo, dataset_path, crop_save_dir)

# 최종 확인
saved_final = set(os.listdir(crop_save_dir))
print(f"\n최종 저장 수: {len(saved_final)} / {len(df_all)}")

- 전처리된 crop 이미지 저장 완료 (14,996 / 15,000).
- 미저장 4개는 single 별도 저장분과의 중복.
- 매핑 실패 277개는 pilliot_15k_v1_final.zip에 미포함된 combination 이미지.

# **3. Data Loader 생성**


In [ ]:
class PillDataset(Dataset):
    def __init__(self, df, crop_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.crop_dir = crop_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.crop_dir, row['image_file'])
        img = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        labels = {
            'drug_shape': torch.tensor(row['shape_group_unified_id'], dtype=torch.long),
            'color_class1': torch.tensor(row['color_group_unified_id'], dtype=torch.long),
        }
        return img, labels

In [ ]:
# transform 정의
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0),
    # transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
crop_dir = '/content/drive/MyDrive/ToBigs/2425/Pillot/dataset/jiwoo/cropped_images/'

saved = set(os.listdir(crop_dir))
df_train_clean = df_train[df_train['image_file'].isin(saved)].copy()
df_val_clean = df_val[df_val['image_file'].isin(saved)].copy()
print(f"train: {len(df_train_clean)} / val: {len(df_val_clean)}")

train_dataset = PillDataset(df_train_clean, crop_dir, transform=train_transform)
val_dataset = PillDataset(df_val_clean, crop_dir, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"train: {len(train_dataset)}장 / val: {len(val_dataset)}장")

In [ ]:
print("train color 분포:")
print(df_train_clean['color_group_unified'].value_counts(normalize=True).round(3))
print("\nval color 분포:")
print(df_val_clean['color_group_unified'].value_counts(normalize=True).round(3))

print("\ntrain shape 분포:")
print(df_train_clean['shape_group_unified'].value_counts(normalize=True).round(3))
print("\nval shape 분포:")
print(df_val_clean['shape_group_unified'].value_counts(normalize=True).round(3))

#### **데이터 로더 구성 요약**
- 전처리된 crop 이미지 기반 PillDataset 구성 (실시간 전처리 대비 학습 속도 대폭 향상)
- 저장된 이미지 기준으로 missing 제외 → train: 18,897장 / val: 3,512장
- train: Resize(224×224) + GaussianBlur + Normalize / val: Resize + Normalize (augmentation 미적용)
- ColorJitter 미적용 — color_group_unified가 분류 타겟이므로 색상 정보 왜곡 방지
- batch_size=32, num_workers=2, pin_memory=True

#### **분포 이슈**
- color: val에 파랑·기타 클래스 누락, 하양 비율 train 33% → val 41%로 쏠림
- shape: 기타 비율 train 12% → val 2.6%로 급감, 원형 train 29% → val 43%로 쏠림
- optimized leakage-free split이 leakage 제거를 우선하다 보니 train/val 간 속성 분포 불균형 발생
- 추후 color/shape 동시 stratify split 또는 희귀 클래스 통합으로 개선 검토 필요

# **4. ConvNeXt-Tiny**

### **모델 아키텍처**

- backbone: ConvNeXt-Tiny (ImageNet1K pretrained), feature extractor로 활용, 출력 feature dim 768
- head: shape_group_unified (4클래스) / color_group_unified (10클래스) 멀티 헤드 구성
- Optimizer: AdamW (lr=1e-4, weight_decay=0.05), CosineAnnealingLR (T_max=10)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device: {device}")

# 모델 정의
backbone = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
backbone.classifier = nn.Identity()

all_num_classes = {
    'drug_shape': 4,
    'color_class1': 10,
}

class PillClassifier(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        feature_dim = 768
        self.heads = nn.ModuleDict({
            name: nn.Linear(feature_dim, n)
            for name, n in num_classes.items()
        })

    def forward(self, x):
        feat = self.backbone(x)
        feat = feat.flatten(1)
        return {name: head(feat) for name, head in self.heads.items()}

model = PillClassifier(backbone, all_num_classes).to(device)
print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 학습 설정
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = {k: 0 for k in model.heads.keys()}
    total = 0

    for i, (imgs, labels) in enumerate(loader):
        imgs = imgs.to(device)
        labels = {k: v.to(device) for k, v in labels.items() if k in model.heads}

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = sum(criterion(outputs[k], labels[k]) for k in model.heads.keys())
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total += imgs.size(0)
        for k in model.heads.keys():
            correct[k] += (outputs[k].argmax(1) == labels[k]).sum().item()

        if i % 10 == 0:
            acc_str = ' | '.join([f"{k}: {correct[k]/total:.4f}" for k in model.heads.keys()])
            print(f"  [train] batch {i}/{len(loader)} | loss: {total_loss/(i+1):.4f} | {acc_str}", flush=True)

    return total_loss / len(loader), {k: correct[k] / total for k in model.heads.keys()}


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = {k: 0 for k in model.heads.keys()}
    total = 0

    with torch.no_grad():
        for i, (imgs, labels) in enumerate(loader):
            imgs = imgs.to(device)
            labels = {k: v.to(device) for k, v in labels.items() if k in model.heads}
            outputs = model(imgs)
            loss = sum(criterion(outputs[k], labels[k]) for k in model.heads.keys())
            total_loss += loss.item()
            total += imgs.size(0)
            for k in model.heads.keys():
                correct[k] += (outputs[k].argmax(1) == labels[k]).sum().item()

            if i % 10 == 0:
                acc_str = ' | '.join([f"{k}: {correct[k]/total:.4f}" for k in model.heads.keys()])
                print(f"  [val]   batch {i}/{len(loader)} | loss: {total_loss/(i+1):.4f} | {acc_str}", flush=True)

    return total_loss / len(loader), {k: correct[k] / total for k in model.heads.keys()}


# 학습 실행
num_epochs = 10
best_val_loss = float('inf')
save_dir = '/content/drive/MyDrive/ToBigs/2425/Pillot/dataset/jiwoo/'

for epoch in range(num_epochs):
    print(f"\n[Epoch {epoch+1:02d}/{num_epochs}]")
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    train_acc_str = ' | '.join([f"{k}: {v:.4f}" for k, v in train_acc.items()])
    val_acc_str = ' | '.join([f"{k}: {v:.4f}" for k, v in val_acc.items()])
    print(f"  train loss: {train_loss:.4f} | {train_acc_str}")
    print(f"  val   loss: {val_loss:.4f} | {val_acc_str}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), save_dir + 'best_model.pth')
        print(f"  best model saved")

epoch 2 학습 중 런타임 끊김으로 중단.

이후 학습 재개 시도했으나 배치 591개로 인한 과도한 학습 시간 문제로 overall 실험 종료.

#### **학습 이슈**
- train 18,897장 기준 batch 591개로 epoch당 학습 시간 과다 소요
- 전처리 이미지 사용에도 불구하고 epoch 1은 Google Drive I/O 병목으로 약 30~35분 소요
- epoch 2부터는 OS 페이지 캐시 효과로 속도 개선되나, 런타임 재시작 시 캐시 초기화로 매번 반복
- 근본 해결책: 학습 시작 전 crop 이미지를 코랩 로컬 디스크(/content/)로 복사하여 Drive I/O 제거 필요

#### **실험 요약**

- single+combination 통합 셋(18,897장)으로 확장 시도 → 배치 591개, epoch당 학습 시간 과다 및 런타임 끊김 문제 발생
- combination crop 품질 문제로 오히려 학습 성능 저하 가능성 있음
- 결론: 추후 데이터 활용/split 전략 개선 검토

